In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import math

import joblib
import numpy as np
import pandas as pd
from scipy import sparse

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# ============================================================
# 1. PATHS
# ============================================================

ARTIFACTS_DIR = Path("artifacts")
OUTPUT_DIR = ARTIFACTS_DIR / "05_feature_engineering"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = ARTIFACTS_DIR / "03_train.csv"
VAL_PATH = ARTIFACTS_DIR / "03_validation.csv"
TEST_PATH = ARTIFACTS_DIR / "03_test.csv"


# ============================================================
# 2. LOAD SPLITS
# ============================================================

print("Loading train / validation / test...")

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)


# ============================================================
# 3. PREDICTION CONTRACT
# ============================================================

# Prediction is made at order purchase time.
# Therefore, future information must not be used as a feature.

TARGET = "is_late"

LEAKAGE_COLUMNS = [
    "order_delivered_customer_date",
    "delivery_delay_days",
    "order_delivered_carrier_date",
    "order_approved_at",
    "order_status",
]

ID_COLUMNS = [
    "order_id",
    "customer_id",
    "customer_unique_id",
]

# ZIP prefix is a location identifier, not a continuous number.
# We already have state + latitude/longitude, so raw ZIP is dropped
# to avoid unnecessary high-cardinality encoding.
DROP_LOCATION_IDENTIFIER_COLUMNS = [
    "customer_zip_code_prefix",
]


# ============================================================
# 4. FEATURE ENGINEERING FUNCTIONS
# ============================================================

DATE_COLUMNS = [
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
]


def haversine_distance_km(
    lat1: pd.Series,
    lon1: pd.Series,
    lat2: pd.Series,
    lon2: pd.Series,
) -> pd.Series:
    """
    Calculate approximate great-circle distance in kilometers
    between customer and seller coordinates.
    """
    earth_radius_km = 6371.0088

    lat1_rad = np.radians(lat1.astype(float))
    lon1_rad = np.radians(lon1.astype(float))
    lat2_rad = np.radians(lat2.astype(float))
    lon2_rad = np.radians(lon2.astype(float))

    delta_lat = lat2_rad - lat1_rad
    delta_lon = lon2_rad - lon1_rad

    a = (
        np.sin(delta_lat / 2) ** 2
        + np.cos(lat1_rad)
        * np.cos(lat2_rad)
        * np.sin(delta_lon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return earth_radius_km * c


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build only features that are available at purchase time.
    """
    result = df.copy()

    # --------------------------------------------------------
    # Parse dates
    # --------------------------------------------------------

    for column in DATE_COLUMNS:
        if column in result.columns:
            result[column] = pd.to_datetime(
                result[column],
                format="mixed",
                errors="coerce",
            )

    # --------------------------------------------------------
    # Purchase-time features
    # --------------------------------------------------------

    purchase = result["order_purchase_timestamp"]

    result["purchase_year"] = purchase.dt.year
    result["purchase_month"] = purchase.dt.month
    result["purchase_day"] = purchase.dt.day
    result["purchase_weekday"] = purchase.dt.day_of_week
    result["purchase_hour"] = purchase.dt.hour
    result["purchase_is_weekend"] = (
        purchase.dt.day_of_week >= 5
    ).astype("int8")

    # --------------------------------------------------------
    # Estimated delivery window
    # --------------------------------------------------------

    result["estimated_delivery_window_days"] = (
        result["order_estimated_delivery_date"]
        - result["order_purchase_timestamp"]
    ).dt.total_seconds() / 86400.0

    # --------------------------------------------------------
    # Geographic distance
    # --------------------------------------------------------

    required_geo_columns = [
        "customer_lat",
        "customer_lng",
        "avg_seller_lat",
        "avg_seller_lng",
    ]

    if all(column in result.columns for column in required_geo_columns):
        result["customer_seller_distance_km"] = haversine_distance_km(
            result["customer_lat"],
            result["customer_lng"],
            result["avg_seller_lat"],
            result["avg_seller_lng"],
        )

    # --------------------------------------------------------
    # Log features for strongly right-skewed variables
    # --------------------------------------------------------

    skewed_columns = [
        "total_item_price",
        "avg_item_price",
        "max_item_price",
        "total_freight_value",
        "avg_product_weight_g",
        "total_payment_value",
    ]

    for column in skewed_columns:
        if column in result.columns:
            result[f"log1p_{column}"] = np.log1p(
                result[column].clip(lower=0)
            )

    # --------------------------------------------------------
    # Drop raw dates after extracting useful features
    # --------------------------------------------------------

    result = result.drop(
        columns=[
            column
            for column in DATE_COLUMNS
            if column in result.columns
        ],
        errors="ignore",
    )

    # --------------------------------------------------------
    # Remove leakage, IDs, and raw ZIP identifier
    # --------------------------------------------------------

    result = result.drop(
        columns=[
            *LEAKAGE_COLUMNS,
            *ID_COLUMNS,
            *DROP_LOCATION_IDENTIFIER_COLUMNS,
        ],
        errors="ignore",
    )

    return result


# ============================================================
# 5. APPLY DETERMINISTIC FEATURE ENGINEERING
# ============================================================

train_engineered = engineer_features(train_df)
val_engineered = engineer_features(val_df)
test_engineered = engineer_features(test_df)

print("\nEngineered shapes:")
print("Train:", train_engineered.shape)
print("Validation:", val_engineered.shape)
print("Test:", test_engineered.shape)


# ============================================================
# 6. SEPARATE X AND y
# ============================================================

y_train = train_engineered[TARGET].astype("int8").to_numpy()
y_val = val_engineered[TARGET].astype("int8").to_numpy()
y_test = test_engineered[TARGET].astype("int8").to_numpy()

X_train = train_engineered.drop(columns=[TARGET])
X_val = val_engineered.drop(columns=[TARGET])
X_test = test_engineered.drop(columns=[TARGET])


# ============================================================
# 7. VALIDATE COLUMN CONSISTENCY
# ============================================================

assert list(X_train.columns) == list(X_val.columns), (
    "Train and validation columns do not match."
)

assert list(X_train.columns) == list(X_test.columns), (
    "Train and test columns do not match."
)

print("\nFeature columns are consistent across all splits.")


# ============================================================
# 8. IDENTIFY NUMERICAL AND CATEGORICAL FEATURES
# ============================================================

numeric_features = X_train.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

unknown_features = [
    column
    for column in X_train.columns
    if column not in numeric_features
    and column not in categorical_features
]

if unknown_features:
    raise ValueError(
        f"Unsupported feature dtypes found: {unknown_features}"
    )

print("\nFeature groups:")
print("Numerical:", len(numeric_features))
print("Categorical:", len(categorical_features))


# ============================================================
# 9. PREPROCESSING PIPELINES
# ============================================================

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="unknown",
            ),
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=20,
                sparse_output=True,
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features,
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features,
        ),
    ],
    remainder="drop",
    sparse_threshold=1.0,
)


# ============================================================
# 10. FIT ON TRAIN ONLY
# ============================================================

print("\nFitting preprocessing pipeline on TRAIN only...")

X_train_transformed = preprocessor.fit_transform(X_train)

print("TRAIN fit complete.")


# ============================================================
# 11. APPLY SAME FITTED PIPELINE TO VALIDATION AND TEST
# ============================================================

print("Transforming validation...")

X_val_transformed = preprocessor.transform(X_val)

print("Transforming test...")

X_test_transformed = preprocessor.transform(X_test)


# ============================================================
# 12. FORCE SPARSE MATRICES
# ============================================================

X_train_transformed = sparse.csr_matrix(X_train_transformed)
X_val_transformed = sparse.csr_matrix(X_val_transformed)
X_test_transformed = sparse.csr_matrix(X_test_transformed)


# ============================================================
# 13. FEATURE NAMES
# ============================================================

feature_names = preprocessor.get_feature_names_out().tolist()

print("\nFinal transformed feature count:", len(feature_names))

print("\nFinal matrix shapes:")
print("Train:", X_train_transformed.shape)
print("Validation:", X_val_transformed.shape)
print("Test:", X_test_transformed.shape)


# ============================================================
# 14. VALIDATION CHECKS
# ============================================================

assert X_train_transformed.shape[0] == len(y_train)
assert X_val_transformed.shape[0] == len(y_val)
assert X_test_transformed.shape[0] == len(y_test)

assert X_train_transformed.shape[1] == X_val_transformed.shape[1]
assert X_train_transformed.shape[1] == X_test_transformed.shape[1]

assert X_train_transformed.shape[1] == len(feature_names)

assert not np.isnan(X_train_transformed.data).any(), (
    "NaN values remain in train features."
)

assert not np.isnan(X_val_transformed.data).any(), (
    "NaN values remain in validation features."
)

assert not np.isnan(X_test_transformed.data).any(), (
    "NaN values remain in test features."
)

print("\nValidation checks: PASS")


# ============================================================
# 15. SAVE TRANSFORMED FEATURE MATRICES
# ============================================================

sparse.save_npz(
    OUTPUT_DIR / "05_train_features.npz",
    X_train_transformed,
)

sparse.save_npz(
    OUTPUT_DIR / "05_validation_features.npz",
    X_val_transformed,
)

sparse.save_npz(
    OUTPUT_DIR / "05_test_features.npz",
    X_test_transformed,
)


# ============================================================
# 16. SAVE LABELS
# ============================================================

np.save(
    OUTPUT_DIR / "05_train_labels.npy",
    y_train,
)

np.save(
    OUTPUT_DIR / "05_validation_labels.npy",
    y_val,
)

np.save(
    OUTPUT_DIR / "05_test_labels.npy",
    y_test,
)


# ============================================================
# 17. SAVE FITTED PREPROCESSOR
# ============================================================

joblib.dump(
    preprocessor,
    OUTPUT_DIR / "05_preprocessor.joblib",
)


# ============================================================
# 18. SAVE FEATURE LIST
# ============================================================

with open(
    OUTPUT_DIR / "05_feature_names.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        feature_names,
        file,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 19. SAVE FEATURE ENGINEERING METADATA
# ============================================================

metadata = {
    "prediction_time": "order_purchase_timestamp",
    "target": TARGET,
    "train_rows": int(X_train_transformed.shape[0]),
    "validation_rows": int(X_val_transformed.shape[0]),
    "test_rows": int(X_test_transformed.shape[0]),
    "final_feature_count": int(X_train_transformed.shape[1]),
    "numeric_feature_count_before_encoding": len(numeric_features),
    "categorical_feature_count_before_encoding": len(
        categorical_features
    ),
    "removed_leakage_columns": LEAKAGE_COLUMNS,
    "removed_id_columns": ID_COLUMNS,
    "removed_location_identifiers": (
        DROP_LOCATION_IDENTIFIER_COLUMNS
    ),
    "created_features": [
        "purchase_year",
        "purchase_month",
        "purchase_day",
        "purchase_weekday",
        "purchase_hour",
        "purchase_is_weekend",
        "estimated_delivery_window_days",
        "customer_seller_distance_km",
        "log1p_total_item_price",
        "log1p_avg_item_price",
        "log1p_max_item_price",
        "log1p_total_freight_value",
        "log1p_avg_product_weight_g",
        "log1p_total_payment_value",
    ],
    "fit_policy": "All fitted preprocessing objects were fit on train only.",
}

with open(
    OUTPUT_DIR / "05_feature_engineering_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 20. SUMMARY
# ============================================================

print("\n" + "=" * 90)
print("NOTEBOOK 5 RESULT")
print("=" * 90)

print("SUCCESS")

print("\nPrediction time:")
print("  order_purchase_timestamp")

print("\nLeakage removed:")
for column in LEAKAGE_COLUMNS:
    print(f"  - {column}")

print("\nArtifacts saved to:")
print(f"  {OUTPUT_DIR}")

print("\nArtifacts:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f"  - {path.name}")

print("\nNotebook 5 completed successfully.")
